## Data Product: Inventory Projections
**Domain:** Supply  
**Target Schema:** `electroniz_catalog.electroniz_gold_supply_schema`  
**Sources:**  
- `electroniz_catalog.electroniz_silver_schema.electroniz_silver_store_orders`  
- `electroniz_catalog.electroniz_silver_schema.electroniz_silver_ecomm_transactions`  
- `electroniz_catalog.electroniz_silver_schema.electroniz_silver_products`  
- `electroniz_catalog.electroniz_silver_schema.electroniz_silver_inventory`  

**Mode:** Continuous Streaming  
**Description:** Estimates inventory projections for the next month and compares against available inventory to identify products requiring restocking.

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW electroniz_product_inventory_projections (
  product_name COMMENT 'Name of the product as defined in the product catalog, used as the primary identifier across sales channels and inventory records',
  product_category COMMENT 'Product classification category grouping related products for inventory planning and demand analysis',
  avg_monthly_demand COMMENT 'Average number of units sold per month across all sales channels (store and ecommerce), calculated from historical order data',
  stddev_demand COMMENT 'Standard deviation of monthly unit sales, measuring demand variability used to calculate safety stock requirements',
  peak_monthly_demand COMMENT 'Highest number of units sold in any single month, representing maximum historical demand for capacity planning',
  min_monthly_demand COMMENT 'Lowest number of units sold in any single month, representing minimum historical demand baseline',
  months_of_data COMMENT 'Number of distinct months with recorded sales data, indicating the breadth of historical demand used in projections',
  predicted_inventory_needed COMMENT 'Projected inventory units required for next month, calculated as average demand plus 1.5x standard deviation safety stock for approximately 93 percent service level',
  current_stock COMMENT 'Most recent available inventory count for the product from the latest inventory snapshot date',
  stock_surplus_or_deficit COMMENT 'Difference between current stock and predicted inventory needed; positive values indicate surplus, negative values indicate a shortfall requiring restocking',
  stock_status COMMENT 'Inventory health classification: SUFFICIENT if current stock covers predicted need, LOW if stock covers average demand but not safety stock, or CRITICAL if stock is below average demand requiring immediate reorder'
)
COMMENT 'Gold-layer materialized view projecting next-month inventory requirements per product by analyzing historical demand from store orders and ecommerce transactions. Compares projected needs against current inventory levels to classify stock health and identify restocking priorities. Part of the Inventory Projections data product in the supply domain.'
TBLPROPERTIES (
  'quality' = 'gold',
  'domain' = 'supply',
  'data_product' = 'inventory projections'
)
AS
WITH product_map AS (
  SELECT product_code, product_name, product_category
  FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_products
),

store_demand AS (
  SELECT p.product_name,
         p.product_category,
         DATE_TRUNC('month', s.order_date) AS order_month,
         SUM(s.units) AS monthly_units
  FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_store_orders s
  JOIN product_map p ON CAST(s.product_id AS STRING) = p.product_code
  WHERE s.order_mode = 'NEW'
  GROUP BY p.product_name, p.product_category, DATE_TRUNC('month', s.order_date)
),

ecomm_demand AS (
  SELECT product_name,
         product_category,
         DATE_TRUNC('month', order_date) AS order_month,
         COUNT(*) AS monthly_units
  FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_ecomm_transactions
  GROUP BY product_name, product_category, DATE_TRUNC('month', order_date)
),

combined_demand AS (
  SELECT product_name, product_category, order_month, monthly_units FROM store_demand
  UNION ALL
  SELECT product_name, product_category, order_month, monthly_units FROM ecomm_demand
),

monthly_totals AS (
  SELECT product_name,
         product_category,
         order_month,
         SUM(monthly_units) AS total_monthly_units
  FROM combined_demand
  GROUP BY product_name, product_category, order_month
),

demand_stats AS (
  SELECT product_name,
         product_category,
         ROUND(AVG(total_monthly_units), 0) AS avg_monthly_demand,
         ROUND(STDDEV(total_monthly_units), 0) AS stddev_demand,
         MAX(total_monthly_units) AS peak_monthly_demand,
         MIN(total_monthly_units) AS min_monthly_demand,
         COUNT(*) AS months_of_data
  FROM monthly_totals
  GROUP BY product_name, product_category
),

current_inventory AS (
  SELECT product_name,
         inventory AS current_stock
  FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_inventory
  WHERE inventory_date = (
    SELECT MAX(inventory_date) FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_inventory
  )
)

SELECT d.product_name,
       d.product_category,
       d.avg_monthly_demand,
       d.stddev_demand,
       d.peak_monthly_demand,
       d.min_monthly_demand,
       d.months_of_data,
       ROUND(d.avg_monthly_demand + 1.5 * COALESCE(d.stddev_demand, 0), 0) AS predicted_inventory_needed,
       COALESCE(i.current_stock, 0) AS current_stock,
       COALESCE(i.current_stock, 0) - ROUND(d.avg_monthly_demand + 1.5 * COALESCE(d.stddev_demand, 0), 0) AS stock_surplus_or_deficit,
       CASE
         WHEN COALESCE(i.current_stock, 0) >= ROUND(d.avg_monthly_demand + 1.5 * COALESCE(d.stddev_demand, 0), 0) THEN 'SUFFICIENT'
         WHEN COALESCE(i.current_stock, 0) >= d.avg_monthly_demand THEN 'LOW - Order Soon'
         ELSE 'CRITICAL - Reorder Now'
       END AS stock_status
FROM demand_stats d
LEFT JOIN current_inventory i ON UPPER(d.product_name) = UPPER(i.product_name)
ORDER BY stock_surplus_or_deficit ASC